# HTML embeddings with Word2Vec
<br>

> #### Gino Prasad
> #### 09/10/2024
<br>


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import math

torch.manual_seed(1)


In [4]:
from gensim.models import Word2Vec
import gensim
from nltk.tokenize import sent_tokenize, word_tokenize
import warnings

In [5]:
# from gensim.test.utils import common_texts
# from gensim.models import Word2Vec
# model = Word2Vec(sentences=common_texts, vector_size=100, window=5, min_count=1, workers=4)
# model.save("word2vec.model")

import gensim.downloader
print(list(gensim.downloader.info()['models'].keys()))
glove_vectors = gensim.downloader.load('glove-wiki-gigaword-300')

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


In [6]:
from nltk.corpus import words
words =  words.words()
common_words = set([x for x in words if glove_vectors.has_index_for(x)])

In [ ]:
common_words

In [40]:
from bs4 import BeautifulSoup
import numpy as np

In [11]:
html_path = "/Users/ginoprasad/Job_Applications/web_crawler/log/09-10_19-59-41.html"
with open(html_path) as infile:
    soup = BeautifulSoup(infile.read(), 'html.parser')

In [295]:
def preprocess_text(string):
    for char in set(string.lower()) - set([chr(x) for x in range(ord('a'), ord('z')+1)]):
        string = string.replace(char, ' ')

    for char in set([chr(x) for x in range(ord('A'), ord('Z')+1)]):
        string = string.replace(char, f' {char.lower()}')

    string = ' '.join([x for x in string.split(' ') if x])
    weird_words = {x for x in string.split(' ') if not glove_vectors.has_index_for(x)}
    weird_map = {}

    for word_mix in weird_words:
        best = (float('-inf'), 0, 0)
        for i in range(1, len(word_mix)):
            best = max(best, (int(word_mix[:i] in common_words) + int(word_mix[i:] in common_words), min(i, len(word_mix)-i), i))
        if best[0] == 2:
            weird_map[word_mix] = f"{word_mix[:best[-1]]} {word_mix[best[-1]:]}"

    for word_mix, replacewith in weird_map.items():
        string = string.replace(word_mix, replacewith)

    for word_mix in weird_words - set(weird_map):
        string = string.replace(word_mix, '?')
    return string

def get_text_stack(element, layers=4, word_limit=20):
    embedding = []
    while 'parent' in dir(element) and len(embedding) < layers:
        if element.text and (not embedding or (embedding[-1] != element.text)):
            embedding.append(element.text)
        element = element.parent
    embedding = [preprocess_text(embedding_i).split(' ')[:word_limit-1] for embedding_i in embedding]
    lengths = torch.Tensor([len(x) for x in embedding]).type(torch.int32)
    embedding = [['^'] + embedding_i + list("*" * (word_limit-length-1)) for length, embedding_i in zip(lengths, embedding)]
    return embedding

def get_text_embedding(element):
    embedding = get_text_stack(element)
    embedding = torch.Tensor(np.array([[glove_vectors.get_vector(x) for x in embedding_i] for embedding_i in embedding]))
        
    # https://stackoverflow.com/questions/53403306/how-to-batch-convert-sentence-lengths-to-masks-in-pytorch
    mask = torch.arange(word_limit).expand(len(lengths), word_limit) < lengths.unsqueeze(1)
    return embedding, mask

In [9]:
class TransformerModel(nn.Module):
    def __init__(self, input_length=300, output_length=100):
        super(TransformerModel, self).__init__()
        self.embedding = nn.Linear(input_length, 512)
        self.encoder_layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, batch_first=True)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6, enable_nested_tensor=False)
        self.positional_encoding = PositionalEncoding(512, 20)
        self.decoder = nn.Sequential(
            nn.Linear(512, 128),
            nn.Flatten(start_dim=1),
            nn.Linear(128*4, output_length))
        
    def forward(self, src, mask):
        assert len(src.shape) == 4
        mask = mask.reshape(-1, 20)
        src = self.embedding(src)
        src = src.reshape(-1, 20, 512)
        x = self.transformer_encoder(src, src_key_padding_mask=mask)
        x = x[:,0,:]
        x = x.reshape(-1, 4, 512)
        return self.decoder(x)
    
    def predict(self, src, mask):
        output = self(src, mask)
        return torch.argmax(output)
    
model = TransformerModel()

In [279]:
element = soup.find(id='input-2')
src, mask = get_text_embedding(element)
src = src.unsqueeze(axis=0)
mask = mask.unsqueeze(axis=0)

In [285]:
model.predict(src, mask)

tensor(37)

In [308]:
element = soup.find(id='input-2')

In [309]:
element.text

'YesNo'

In [310]:
[' '.join(x) for x in get_text_stack(element)]

['^ yes no * * * * * * * * * * * * * * * * *',
 '^ have you ever been employed by broadcom or any of the broadcom inc companies yes no * * *',
 '^ how did you hear about us select one have you ever been employed by broadcom or any of the',
 '^ my information indicates a required field how did you hear about us select one have you ever been employed']